# Introduction
This notebook demonstrates how to set up and run a quantized version of the Llama-3-8B model. We will begin with some basic setup and then proceed to load and use the model.

## 1. Initial Setup

In [1]:
print("Hello World")
!nvidia-smi --query-gpu=memory.free --format=csv | tail -n +2 | awk -F ' ' '{print "Free GPU Memory (GB):", $1 / 1024}'

Hello World
Free GPU Memory (GB): 44.5264


In [2]:
print("\n################################")
print("Setting up environment...")
print("################################\n")

import os
#os.chdir('..')
print("Current Working Directory ", os.getcwd())
import sys
sys.path.append("../") # Add directory containing src/data to path

import importlib
import src  # Assuming src is the package name

# Reload the src module after making changes
importlib.reload(src)

%load_ext autoreload
%autoreload 2

os.environ["TOKENIZERS_PARALLELISM"] = "false"  # Disables parallelism to remove transformers warning

print("\n################################")
print("Setting up cache paths...")
print("################################\n")

os.environ["MKL_SERVICE_FORCE_INTEL"] = "1"
CACHE_PATH = "/nfs/students/daro/.cache/huggingface"
print(f"Setting cache path to {CACHE_PATH}")

os.environ["TORCH_HOME"] = CACHE_PATH
os.environ["HF_HOME"] = CACHE_PATH

import torch
torch.hub.set_dir(CACHE_PATH)
with torch.no_grad():
    torch.cuda.empty_cache()
    
!cat /proc/meminfo | awk '/MemTotal/ {total=$2} /MemFree/ {free=$2} /MemAvailable/ {available=$2} END {printf "MemTotal: %.2f GB\nMemFree: %.2f GB\nMemAvailable: %.2f GB\n", total/1024/1024, free/1024/1024, available/1024/1024}'
!nvidia-smi --query-gpu=memory.free --format=csv | tail -n +2 | awk -F ' ' '{print "Free GPU Memory (GB):", $1 / 1024}'

# Code formatting and linting

# !black notebooks/Llama-3-8B-quant.ipynb
# !pylint notebooks/Llama-3-8B-quant.ipynb

print("\n################################")
print("Setting up cuda devices...")
print("################################\n")

if torch.cuda.is_available():
    print("CUDA device is available!")
    # Get the number of available CUDA devices
    num_cuda_devices = torch.cuda.device_count()
    print(f"Number of CUDA devices: {num_cuda_devices}")
    
    # Loop through available devices and get name
    for device_id in range(num_cuda_devices):
        device = torch.device(f"cuda:{device_id}")
        name = torch.cuda.get_device_name(device)
        print(f"  - CUDA Device {device_id+1}: {name}")
else:
    print("CUDA device is not available.")
    
print("\n################################")
print("Authentication with Hugging Face...")
print("################################\n")

import os
from dotenv import load_dotenv
from huggingface_hub import login

load_dotenv()
huggingface_token = os.getenv('HUGGINGFACE_TOKEN')

if huggingface_token is None:
    raise ValueError("Please set the HUGGINGFACE_TOKEN environment variable.")
else:
    print("Hugging Face token loaded successfully.")

login(token=huggingface_token, add_to_git_credential=True)
print("Successfully authenticated with the Hugging Face API.")

print("\n################################")
print("Setting up GPU memory usage list...")
print("################################\n")
# Global list to store GPU memory usage
from src.evaluations.evaluate_memory import record_gpu_memory
gpu_memory_usage = {}
record_gpu_memory(gpu_memory_usage=gpu_memory_usage, context="Warm up notebook")


################################
Setting up environment...
################################

Current Working Directory  /nfs/homedirs/daro/git/quantization-reliability
Initializing src package
Initializing src package

################################
Setting up cache paths...
################################

Setting cache path to /nfs/students/daro/.cache/huggingface
MemTotal: 251.59 GB
MemFree: 14.68 GB
MemAvailable: 247.75 GB
Free GPU Memory (GB): 44.5264

################################
Setting up cuda devices...
################################

CUDA device is available!
Number of CUDA devices: 1
  - CUDA Device 1: NVIDIA L40S

################################
Authentication with Hugging Face...
################################

Hugging Face token loaded successfully.
Token is valid (permission: write).
Your token has been saved in your configured git credential helpers (store).
Your token has been saved to /nfs/students/daro/.cache/huggingface/token
Login successful
Successful

## 2. Loading Model

In [3]:
from transformers import AutoTokenizer

device = "cuda"
model_name = "meta-llama/Meta-Llama-3-8B"  # Too large to run on a gpu_gtx1080. GPU gpu_a100 is required.

tokenizer = AutoTokenizer.from_pretrained(model_name, device_map=device)
tokenizer.pad_token_id = tokenizer.eos_token_id
tokenizer.padding_side = "left"
if tokenizer.model_max_length > 1e6:
  print(f"Tokenizer model max length reduced from {tokenizer.model_max_length} to 2048 to fit in memory")
  tokenizer.model_max_length = 2048

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.


Tokenizer model max length reduced from 1000000000000000019884624838656 to 2048 to fit in memory


In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

device = "cuda"

# model_name = "EleutherAI/gpt-neo-125m"  # Lightweight model for debugging purposes
model_name = "meta-llama/Meta-Llama-3-8B"  # Too large to run on a gpu_gtx1080. GPU gpu_a100 is required.
# model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"  # Small enough to run on a gpu_gtx1080.
# model_name = "openai-community/gpt2-large"

tokenizer = AutoTokenizer.from_pretrained(model_name, device_map=device)
model = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype="auto", device_map="cuda")
# TODO: Check why dtype = auto solved the problem
# TODO: what is the default value of torch_dtype -> look in the githubb documentation
# Always use "auto"
model.NAME = model_name

tokenizer.pad_token_id = tokenizer.eos_token_id
tokenizer.padding_side = "left"

if tokenizer.model_max_length > 1e6:
  print(f"Tokenizer model max length reduced from {tokenizer.model_max_length} to 2048 to fit in memory")
  tokenizer.model_max_length = 2048

!nvidia-smi --query-gpu=memory.free --format=csv | tail -n +2 | awk -F ' ' '{print "Free GPU Memory (GB):", $1 / 1024}'
print(f"Loaded model {model_name} with the following configuration:")
print(f"- model max length: {tokenizer.model_max_length}")
print(f"- dtype: {model.dtype}")
print(f"- device: {model.device}")
print(f"- parameters: {(lambda p: f'{p / 1e9:.1f}B' if p > 1e9 else (f'{p / 1e6:.1f}M' if p > 1e6 else str(p)))(model.num_parameters())}")
print(f"- memory footprint: {model.get_memory_footprint() / (1024 ** 3):.2f} GB")
print(f"- vocabulary size: {tokenizer.vocab_size}")
print(f"- padding token ID: {tokenizer.pad_token_id}")
print(f"- special tokens: {tokenizer.special_tokens_map}")

from src.evaluations.evaluate_memory import record_gpu_memory
record_gpu_memory(gpu_memory_usage=gpu_memory_usage, context="Load model")

Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.
Loading checkpoint shards: 100%|██████████| 4/4 [00:23<00:00,  5.79s/it]


Tokenizer model max length reduced from 1000000000000000019884624838656 to 2048 to fit in memory
Free GPU Memory (GB): 21.6621
Loaded model meta-llama/Meta-Llama-3-8B with the following configuration:
- model max length: 2048
- dtype: torch.bfloat16
- device: cuda:0
- parameters: 8.0B
- memory footprint: 14.96 GB
- vocabulary size: 128000
- padding token ID: 128001
- special tokens: {'bos_token': '<|begin_of_text|>', 'eos_token': '<|end_of_text|>', 'pad_token': '<|end_of_text|>'}
Free GPU Memory (GB): 21.6621. Context: Load model.


## 3. Loading Datasets

### 3.1. WikiText

In [5]:
# Initialize the datamodule
import os
from src.data.WikiTextDataModule import WikiTextDataModule

print("\n################################")
print("Setting up WikiTextDataModule...")
print("################################\n")

wikitext_data_module = WikiTextDataModule(
  directory_dataset=os.getcwd(),
  batch_size=1,
  sequence_length=2048,
  tokenizer_name=model_name,
  seed=3,
  n_lines = None
)

wikitext_dataloader = wikitext_data_module.test_dataloader()

print("\n################################")
print("Printing properties of WikiTextDataModule...")
print("################################\n")

# Print properties
print(f"Length of train dataset: {len(wikitext_data_module.train_dataset)}")
print(f"Length of validation dataset: {len(wikitext_data_module.val_dataset)}")
print(f"Length of test dataset: {len(wikitext_data_module.test_dataset)}")

print("\nTotal number of tokens in each dataset:")
print(f"Train dataset: {sum([len(data_string) for data_string in wikitext_data_module.train_dataset['text']])}")
print(f"Validation dataset: {sum([len(data_string) for data_string in wikitext_data_module.val_dataset['text']])}")
print(f"Test dataset: {sum([len(data_string) for data_string in wikitext_data_module.test_dataset['text']])}")

total_string = "".join([data_string for data_string in wikitext_data_module.val_dataset['text']])
total_string_len = len(total_string)
tokenized_string = tokenizer.encode(total_string, return_tensors="pt")

print(f"\nLength of total validation dataset (characters): {total_string_len}")
print(f"Length of tokenized validation dataset (tokens): {len(tokenized_string[0])}")
print(f"Tokenizer compression rate: {(100 * len(tokenized_string[0]) / total_string_len):.2f}%")

# Reason why the numbers are low: number of tokens / 2048 -> gives the number of elements in the dataset
dataset_size = len(wikitext_dataloader)
print(f"\nNumber of batches in validation dataloader: {dataset_size}")

for i, (data, target) in enumerate(wikitext_dataloader):
    if i < 1:
        print(f"\nBatch {i + 1}:")
        original_text = tokenizer.decode(data[0], skip_special_tokens=True)
        print(f"  Original Text: {original_text[:500]}...")  # Print the first 500 characters
        print(f"  Input data (first 5 tokens): {data[0][:5]}")
        print(f"  Target labels (first 5 tokens): {target[0][:5]}")
        print(f"  Input data shape: {data.shape}")
        print(f"  Target labels shape: {target.shape}")


################################
Setting up WikiTextDataModule...
################################



Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.



################################
Printing properties of WikiTextDataModule...
################################

Length of train dataset: 36718
Length of validation dataset: 3760
Length of test dataset: 4358

Total number of tokens in each dataset:
Train dataset: 10892990
Validation dataset: 1142150
Test dataset: 1285622


Token indices sequence length is longer than the specified maximum sequence length for this model (252726 > 2048). Running this sequence through the model will result in indexing errors



Length of total validation dataset (characters): 1142150
Length of tokenized validation dataset (tokens): 252726
Tokenizer compression rate: 22.13%

Number of batches in validation dataloader: 564

Batch 1:
  Original Text:  The date chosen was 5 October 1968, an adhoc committee was formed ( although in reality most of the organizing was done by McCann and Melaugh ) and the route was to take the marchers inside the city walls, where nationalists were traditionally not permitted to march. The Minister of Home Affairs, William Craig, made an order on 3 October prohibiting the march on the grounds that the Apprentice Boys of Derry were intending to hold a march on the same day. In the words of Martin Melaugh of CAIN "...
  Input data (first 5 tokens): tensor([  578,  2457, 12146,   574,   220])
  Target labels (first 5 tokens): tensor([-100, -100, -100, -100, -100])
  Input data shape: torch.Size([1, 2048])
  Target labels shape: torch.Size([1, 2048])


### 3.2. OpenAssistant

In [ ]:
# Initialize the datamodule
import os
from src.data.OpenAssistantDataModule import OpenAssistantDataModule

# Data Module
oasst_data_module = OpenAssistantDataModule(
  directory_dataset=os.getcwd(),
  batch_size=1,
  sequence_length=2048,
  tokenizer_name=model_name,
  seed=1
)

# Data Loader
# oasst_dataloader = oasst_data_module.train_dataloader()
oasst_dataloader = oasst_data_module.val_dataloader()

print(f"Length of datasets:", len(oasst_data_module.train_dataset), len(oasst_data_module.val_dataset))

# Reason why the numbers are low: number of tokens / 2048 -> gives the number of elements in the dataset

oasst_dataset_size = len(oasst_dataloader)
print(f"Number of batches in train_dataloader: {dataset_size}")
for i, (data, target) in enumerate(oasst_dataloader):
    if i < 2:
        print(f"Batch {i+1}:")
        original_text = tokenizer.decode(data[0], skip_special_tokens=True)
        print(f"  Original Text: {original_text[:500]}")
        print(f"  Input data (first 5 tokens): {data[0][:5]}")
        print(f"  Target labels (first 5 tokens): {target[0][:5]}")
        print(f"  Input data shape: {data.shape}")
        print(f"  Target labels shape: {target.shape}")

## 4. Quantization

### 4.1 GPTQ

In [4]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from optimum.gptq import GPTQQuantizer, load_quantized_model
import torch
model_name = "meta-llama/Meta-Llama-3-8B"
tokenizer = AutoTokenizer.from_pretrained(model_name, device_map="cuda")
model = AutoModelForCausalLM.from_pretrained(model_name, torch_dtype="auto", device_map="cuda")

quantizer = GPTQQuantizer(bits=4, dataset="wikitext2", device="cuda")
quantized_model = quantizer.quantize_model(model, tokenizer)

2024-10-31 09:45:25.322647: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:485] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2024-10-31 09:45:25.332328: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:8454] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2024-10-31 09:45:25.335261: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1452] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2024-10-31 09:45:25.344223: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2024-10-31 09:45:26.827034: W tensorflow/compiler/tf2

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Quantizing model.layers blocks :   0%|          | 0/32 [00:00<?, ?it/s]

Quantizing layers inside the block:   0%|          | 0/7 [00:00<?, ?it/s]

Quantizing layers inside the block:   0%|          | 0/7 [00:00<?, ?it/s]

...

Quantizing layers inside the block:   0%|          | 0/7 [00:00<?, ?it/s]

Quantizing layers inside the block:   0%|          | 0/7 [00:00<?, ?it/s]

Quantizing layers inside the block:   0%|          | 0/7 [00:00<?, ?it/s]

Quantizing layers inside the block:   0%|          | 0/7 [00:00<?, ?it/s]

Using Exllamav2 backend will reorder the weights offline, thus you will not be able to save the model with the right weights.Setting `disable_exllama=True`. You should only use Exllamav2 backend for inference. 
/nfs/students/daro/miniconda3/envs/env-quant-rel-310/lib/python3.10/site-packages/transformers/modeling_utils.py:4481: FutureWarning: `_is_quantized_training_enabled` is going to be deprecated in transformers 4.39.0. Please use `model.hf_quantizer.is_trainable` instead
  warnings.warn(


In [6]:
import os
import torch
from transformers import AutoModelForCausalLM, HqqConfig
from src import MODEL_SAVE_PATH

gptq_model_name = f"{model_name.split('/')[1]}-GPTQ-4"
gptq_model_path = os.path.join(MODEL_SAVE_PATH, gptq_model_name)
quantized_model.NAME = gptq_model_name
quantized_model.PATH = gptq_model_path
!nvidia-smi --query-gpu=memory.free --format=csv | tail -n +2 | awk -F ' ' '{print "Free GPU Memory (GB):", $1 / 1024}'

print("GPTQ model name:", gptq_model_name)
print("GPTQ model path:", gptq_model_path)

Free GPU Memory (GB): 26.4531
GPTQ model name: Meta-Llama-3-8B-GPTQ-4
GPTQ model path: /nfs/students/daro/models/Meta-Llama-3-8B-GPTQ-4


In [7]:
print("\n################################")
print("Saving quantized model...")
print("################################\n")
quantizer.save(quantized_model, gptq_model_path)


################################
Saving quantized model...
################################



## 5. Evaluation

In [ ]:
!nvidia-smi --query-gpu=memory.free --format=csv | tail -n +2 | awk -F ' ' '{print "Free GPU Memory (GB):", $1 / 1024}'

### 5.1. Perplexity

In [20]:
import torch
import torchmetrics

def evaluate_perplexity(model, dataloader, factor=1, device="cuda", to_device=False):
    if to_device:
        model.to(device)
    if isinstance(model, torch.nn.Module):
        model.eval()
        print(f"Model in evaluation mode. Device: {device}")
    with torch.no_grad():
        torch.cuda.empty_cache()
        
    metric = torchmetrics.text.Perplexity(ignore_index=-100).to(device)  # -100 is the padding token.

    for i, (x, y) in enumerate(dataloader):
        if i >= len(dataloader) / factor:
            break
        x, y = x.to(device), y.to(device)
        
        with torch.no_grad():
            outputs = model(x)
            logits = outputs.logits
            
            # Metric on current batch
            perplexity = metric(logits.float(), y)

    # Metric on all batches using custom accumulation
    perplexity = metric.compute()
    print(f"Final Perplexity: {perplexity:.3f}")
    return perplexity.item()

In [21]:
for model_obj in [model, hqq_model_4]:
    wikitext_dataloader = wikitext_data_module.test_dataloader()
    print(f"Calculating perplexity for model {model_obj.NAME}")
    ppl = evaluate_perplexity(model_obj, wikitext_dataloader, factor=1, device="cuda", to_device=False)
    print(f"Perplexity for model {model_obj.NAME}: {ppl:.3f}")

Calculating perplexity for model TinyLlama/TinyLlama-1.1B-Chat-v1.0
Model in evaluation mode. Device: cuda
Final Perplexity: 7.972
Perplexity for model TinyLlama/TinyLlama-1.1B-Chat-v1.0: 7.972
Calculating perplexity for model TinyLlama-1.1B-Chat-v1.0-hqq-lora-v2
Model in evaluation mode. Device: cuda
Final Perplexity: 8.208
Perplexity for model TinyLlama-1.1B-Chat-v1.0-hqq-lora-v2: 8.208


In [ ]:
wikitext_dataloader = wikitext_data_module.test_dataloader()
ppl = evaluate_perplexity(model, wikitext_dataloader, factor= device=device)
print(f"\nFinal Perplexity (PPL): {ppl:.3f}")

In [ ]:
wikitext_dataloader = wikitext_data_module.test_dataloader()
ppl = evaluate_perplexity(model_same, wikitext_dataloader, device=device)
print(f"\nFinal Perplexity (PPL): {ppl:.3f}")

In [ ]:
wikitext_dataloader = wikitext_data_module.test_dataloader()
ppl = evaluate_perplexity(model_dynamic, wikitext_dataloader, device=device)
print(f"\nFinal Perplexity (PPL): {ppl:.3f}")

In [ ]:
import numpy as np
lls = torch.tensor(lls)
print(stride)
print(lls/stride)
print(torch.exp(lls / (stride)))
print(torch.exp(lls.sum() / (31 * stride)))

ppls = [ppl for ppl in ppls]
print(ppls)

print(xs[2])
print(ys[2])
print(input_ids_list[2])
print(target_ids_list[2])

print(outputs[0])
print()

In [ ]:
!nvidia-smi --query-gpu=memory.free --format=csv | tail -n +2 | awk -F ' ' '{print "Free GPU Memory (GB):", $1 / 1024}'

In [ ]:
evaluate_perplexity(model_bnb_8bit, tokenizer, wikitext_data_module, device=device)

In [ ]:
evaluate_perplexity(model_bnb_4bit, tokenizer, wikitext_data_module, device=device)

In [ ]:
evaluate_perplexity(awq_model, wikitext_dataloader, device="cuda")

In [ ]:
list_of_models = [model, model_bnb_8bit, model_bnb_4bit]
results = {}
for model in list_of_models:
  print(f"Perplexity for model {model.NAME}: {evaluate_perplexity(model_bnb_8bit, wikitext_data_module, device)}"

# Print perplexity results
#print(f"Perplexity (8-bit): {perplexity_8bit:.4f}")
#print(f"Perplexity (4-bit): {perplexity_4bit:.4f}")
print(f"Perplexity (Original): {perplexity_original:.4f}")

### 5.2. Brier Score

In [ ]:
import torch
import torch.nn.functional as F
from torch.cuda.amp import autocast

class BrierScore:
    def __init__(self, device="cpu"):
        self.device = device
        self.reset()

    def reset(self):
        self.total_brier_score = 0.0
        self.num_batches = 0

    def update(self, probs, targets):
        brier_score = torch.mean((probs - targets) ** 2)
        self.total_brier_score += brier_score.item()
        self.num_batches += 1

    def compute(self):
        if self.num_batches == 0:
            return 0.0
        return self.total_brier_score / self.num_batches

def evaluate_brier_score(model, dataloader, device="cuda", to_device=False):
    if to_device:
        model.to(device)

    if isinstance(model, torch.nn.Module):
        model.eval()

    print(f"Model in evaluation mode. Device: {device}")
    
    # Initialize BrierScore metric
    metric = BrierScore(device=device)
    
    for i, (x, y) in enumerate(dataloader):
        if i > 10:
            break
        print(f"Processing batch {i}")
        x, y = x.to(device), y.to(device)

        with torch.no_grad() and autocast():
            outputs = model(x)
            logits = outputs.logits
            !nvidia-smi --query-gpu=memory.free --format=csv | tail -n +2 | awk -F ' ' '{print "Free GPU Memory (GB):", $1 / 1024}'

            # Shift logits and target_ids to the left by 1 for calculating the Brier score
            shifted_logits = logits[:, :-1].contiguous()
            shifted_target_ids = x[:, 1:].contiguous()

            # Flatten the logits and target_ids for calculation
            shifted_logits = shifted_logits.view(-1, shifted_logits.size(-1))
            shifted_target_ids = shifted_target_ids.view(-1)

            # Filter out the -100 targets
            valid_indices = shifted_target_ids != -100
            valid_logits = shifted_logits[valid_indices]
            valid_target_ids = shifted_target_ids[valid_indices]

            # Get the probabilities
            probs = F.softmax(valid_logits, dim=-1)

            # Create one-hot target vectors
            targets = F.one_hot(valid_target_ids, num_classes=probs.size(-1)).float()

            # Update the metric with the current batch's results
            metric.update(probs, targets)

    # Compute the final Brier score across all batches
    avg_brier_score = metric.compute()
    print(f"Final Brier Score: {avg_brier_score:.10f}")

    return avg_brier_score

# Assuming wikitext_data_module and model are defined elsewhere
wikitext_dataloader = wikitext_data_module.test_dataloader()
final_brier_score = evaluate_brier_score(model, wikitext_dataloader, device=device)
print(f"\nFinal Brier Score: {final_brier_score:.10f}")

In [ ]:
evaluate_brier_score(model, tokenizer, wikitext_dataloader, factor=100, device=device)

In [ ]:
evaluate_brier_score(model_bnb_4bit, tokenizer, wikitext_data_module, device=device)